In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("..")
os.environ["VLLM_USE_V1"] = "1"

In [3]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.float
device   = 'cuda'
model_id = "Qwen/QwQ-32B"

In [4]:
from pathlib import Path

cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/qwq-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

In [5]:
task_name = "plan_generation_po"
eval_results = [
    load_dataset_from_file(domain_name, task_name)["instances"] for domain_name in [
        "blocksworld_mystery_2",
    ]
]
eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]

In [6]:

tokenizer = initialize_tokenizer(model_id)
dataset = load_dataset(f"dmitriihook/qwq-32b-planning-mystery-2-24k-greedy")["train"]

In [7]:
DOMAIN_PHRASES = {
    "mystery_2": {
        "actions": {
            "attack": "illuminate",
            "succumb": "silence",
            "overcome": "distill",
            "feast": "divest"
        },
        "predicates": {
            "planet": "aura",
            "province": "essence",
            "harmony": "nexus",
            "craves": "harmonizes",
            "pain": "pulse"
        }
    },
}

In [8]:
def extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=True):
    """Find end of the phrase token positions"""
    tokens = tokens.squeeze()

    phrase_tokens = [
        tokenizer.encode(" " + phrase),
        tokenizer.encode(" " + phrase.capitalize()),
        tokenizer.encode("\n" + phrase)[1:],
        tokenizer.encode("\n" + phrase.capitalize())[1:],
        tokenizer.encode("\n\n" + phrase)[1:],
        tokenizer.encode("\n\n" + phrase.capitalize())[1:],
    ]

    positions = set()

    if cot_only:
        start_pos = torch.where(tokens == 151667)[0]
        start_mask = torch.arange(tokens.shape[0]) >= start_pos

    for phts in phrase_tokens:
        presence_mask = torch.ones_like(tokens)
        if cot_only:
            presence_mask = presence_mask * start_mask

        for i, t in enumerate(phts):
            presence_mask = presence_mask * (tokens == t)[i:]
            presence_mask = presence_mask[:-1]

        for p in (torch.where(presence_mask)[0]).tolist():
            positions.add(
                tuple([p-1, p + len(phts)])
            )        
    
    return sorted(list(set(positions)))

In [9]:
from vllm import LLM

llm = LLM(model=model_id, tensor_parallel_size=4, enforce_eager=True, max_seq_len_to_capture=20000, max_num_batched_tokens=4096, enable_prefix_caching=False)

INFO 04-11 13:46:14 [__init__.py:239] Automatically detected platform cuda.
INFO 04-11 13:46:23 [config.py:600] This model supports multiple tasks: {'reward', 'score', 'classify', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 04-11 13:46:24 [config.py:1600] Defaulting to use mp for distributed inference
INFO 04-11 13:46:24 [config.py:1780] Chunked prefill is enabled with max_num_batched_tokens=4096.
WARNING 04-11 13:46:24 [cuda.py:96] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 04-11 13:46:25 [core.py:61] Initializing a V1 LLM engine (v0.8.3) with config: model='Qwen/QwQ-32B', speculative_config=None, tokenizer='Qwen/QwQ-32B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=4, pipeline

Loading safetensors checkpoint shards:   0% Completed | 0/14 [00:00<?, ?it/s]


(VllmWorker rank=3 pid=395843) INFO 04-11 13:46:47 [loader.py:447] Loading weights took 6.42 seconds
(VllmWorker rank=3 pid=395843) INFO 04-11 13:46:47 [gpu_model_runner.py:1273] Model loading took 15.3937 GiB and 8.074196 seconds
(VllmWorker rank=1 pid=395786) INFO 04-11 13:46:48 [loader.py:447] Loading weights took 6.64 seconds
(VllmWorker rank=1 pid=395786) INFO 04-11 13:46:48 [gpu_model_runner.py:1273] Model loading took 15.3937 GiB and 8.824399 seconds
(VllmWorker rank=0 pid=395760) INFO 04-11 13:46:48 [loader.py:447] Loading weights took 6.91 seconds
(VllmWorker rank=2 pid=395813) INFO 04-11 13:46:48 [loader.py:447] Loading weights took 6.55 seconds
(VllmWorker rank=0 pid=395760) INFO 04-11 13:46:48 [gpu_model_runner.py:1273] Model loading took 15.3937 GiB and 9.383054 seconds
(VllmWorker rank=2 pid=395813) INFO 04-11 13:46:49 [gpu_model_runner.py:1273] Model loading took 15.3937 GiB and 9.522867 seconds
INFO 04-11 13:46:54 [kv_cache_utils.py:578] GPU KV cache size: 658,240 token

In [10]:
with open(
    "mean_reprs_mystery_2.json",
    'r'
) as f:
    reprs = json.load(f)

In [11]:
mean_reprs = {
    k: np.array(v) for k, v in reprs["mean_reprs"].items()
}

In [12]:
mean_actions = np.array(reprs["mean_domain"])
mean_predicates = np.array(reprs["mean_domain"])

In [13]:
phrases = DOMAIN_PHRASES["mystery_2"]
phrases = list(phrases["actions"].values()) + list(phrases["predicates"].values())

In [14]:
# phrase_positions = {
#     phrase: extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=False)
#     for phrase in phrases
# }

In [15]:
# pharse_masks = {
#     phrase: np.zeros(tokens.shape[0])
#     for phrase in phrases
# }

# for ip, phrase in enumerate(phrases):
#     positions = phrase_positions[phrase]
    
#     for start, end in positions:
#         pharse_masks[phrase][start:end] = 1


In [16]:

# masks_batch = {
#     k: [v, v] for k, v in pharse_masks.items()
# }

# masks_batch_combined = {
#     k: np.concatenate(v, axis=0) for k, v in masks_batch.items()
# }

# combined_len = masks_batch_combined[phrases[0]].shape[0]

In [17]:
log_path = cur_dir / "vllm_log.log"

In [18]:
def logging_hook(module, input, output):
    # return
    with open(log_path, 'a') as f:
        a = module._meta.get("a", 0)
        module._meta["a"] = a + 1
        # f.write(str(input[0].tolist()) + "\n")
        if input[0].shape[0] > 10:
            f.write(str(input[1]) + "|" + str(a) + "\n")
        # f.write(str(input[1].shape) + "\n")
        # f.flush()

In [19]:
from collections import OrderedDict

def add_hook(module, hook):
    module._forward_hooks = OrderedDict()
    module._meta = {}
    module.register_forward_hook(hook)


In [20]:

def f(x):
    add_hook(
        x.worker.model_runner.model.model.layers[47],
        logging_hook
    )

res = llm.llm_engine.collective_rpc(
    f,
)

res

[None, None, None, None]

In [21]:
from vllm import TokensPrompt
all_tokens = []
phrase_masks = {phrase: [] for phrase in phrases}

for i in [0,0,0,0]:

    row = dataset[i]

    text = "\n\n".join(row["generation"].split("\n\n")[:40])
    tokens = tokenize_blocksworld_generation(tokenizer, row, text)[:, :-2][0]
    
    all_tokens.append(tokens)
    
    # Get phrase positions for this row
    phrase_positions = {
        phrase: extract_all_phrase_positions(tokens, phrase, tokenizer, cot_only=False)
        for phrase in phrases
    }
    
    # Create phrase masks for this row
    row_phrase_masks = {
        phrase: np.zeros(tokens.shape[0])
        for phrase in phrases
    }
    
    for phrase in phrases:
        positions = phrase_positions[phrase]
        for start, end in positions:
            row_phrase_masks[phrase][start:end] = 1
    
    # Add masks to batch
    for phrase in phrases:
        phrase_masks[phrase].append(row_phrase_masks[phrase])

masks_combined = {
        k: np.concatenate(v, axis=0) for k, v in phrase_masks.items()
    }
    
combined_len = masks_combined[phrases[0]].shape[0] if phrases else 0

In [22]:
from collections import OrderedDict

block_size = 4096

def hook(module, input, output):
    try:
        meta = getattr(module, "_meta", {})
        meta["mask_offset"] = meta.get("mask_offset", 0)
        
        if meta["mask_offset"] >= combined_len:
            return output
        
        mask_start = meta["mask_offset"]
        mask_end = mask_start + block_size
        
        meta["mask_offset"] = mask_end
        module._meta = meta
        
        hs, res = output
        
        for ip, phrase in enumerate(phrases):
            if ip < 4:
                adjustment = mean_actions
            else:
                adjustment = mean_predicates
            
            steering_mask = masks_combined[phrase]
            
            steering_mask = steering_mask[mask_start:mask_end]
            steering_mask = np.concatenate([steering_mask, np.zeros(hs.shape[0] - steering_mask.shape[0])], axis=0)
            
            steering_vector = mean_reprs[phrase] - adjustment
            steering_vector = steering_mask[:, None] * steering_vector
            steering_vector = torch.tensor(steering_vector, dtype=hs.dtype, device=hs.device)
        
            hs += steering_vector * 1
        
        return hs, res
    except Exception as e:
        with open(log_path, 'a') as f:
            f.write(str(e) + "\n")
        raise e

In [23]:
def logging_hook(module, input, output):
    meta = getattr(module, "_meta", {})
    meta["mask_offset_2"] = meta.get("mask_offset_2", 0)
    
    if meta["mask_offset_2"] >= combined_len:
        return output
    
    mask_start = meta["mask_offset_2"]
    mask_end = mask_start + block_size
    
    meta["mask_offset_2"] = mask_end
    module._meta = meta
    
    hs, res = output
    
    torch.save(
        {
            "hs": hs.cpu(),
            "res": res.cpu()
        },
        f"vllm_hidden_states_v1/0_{mask_start}_{mask_end}_clean.pt"
    )


In [24]:
from vllm import TokensPrompt

prompts = [
    TokensPrompt(
        prompt_token_ids=x.tolist(),
    ) for x in all_tokens
]

In [25]:

def add_hook(module, hooks):
    module._forward_hooks = OrderedDict()
    module._meta = {}
    for hook in hooks:
        module.register_forward_hook(hook)

def empty_hook(module, input, output):
    pass


def f(x):
    model = x.worker.model_runner.model.model
    add_hook(model.layers[47], [empty_hook])
    add_hook(model.layers[48], [logging_hook])


res = llm.llm_engine.collective_rpc(
    f,
)

In [26]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    max_tokens=5,
    temperature=0,
    top_k=1,
)

In [28]:
res1 = llm.generate(
    prompts, sampling_params=sampling_params
)

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 4/4 [00:00<00:00,  5.28it/s, est. speed input: 10490.59 toks/s, output: 26.41 toks/s]


In [29]:
dumped_hs = {}
dumped_res = {}

for path in Path("vllm_hidden_states_v1").glob("*.pt"):
    with open(path, 'rb') as f:
        data = torch.load(f)
        dumped_hs[path.stem] = data["hs"]
        dumped_res[path.stem] = data["res"]

In [30]:
hs_clean = dumped_hs["0_0_4096_clean"]
hs_clean_2 = hs_clean[len(tokens):len(tokens) + len(tokens)]

res_clean = dumped_res["0_0_4096_clean"]
res_clean_2 = res_clean[len(tokens):len(tokens) + len(tokens)]

hs = dumped_hs["0_0_4096"]
hs_2 = hs[len(tokens):len(tokens) + len(tokens)]

res = dumped_res["0_0_4096"]
res_2 = res[len(tokens):len(tokens) + len(tokens)]

res.shape

torch.Size([4096, 5120])

In [31]:
hs_clean = dumped_hs["0_0_4096_clean"]
hs_clean_1 = hs_clean[:len(tokens)]

res_clean = dumped_res["0_0_4096_clean"]
res_clean_1 = res_clean[:len(tokens)]

hs = dumped_hs["0_0_4096"]
hs_1 = hs[:len(tokens)]

res = dumped_res["0_0_4096"]
res_1 = res[:len(tokens)]

In [ ]:
(res_1 - res_clean_1)[:23]

tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.1445,  0.2539, -0.3828,  ...,  3.3750,  0.0625,  0.3125],
        [ 0.0938,  0.3477,  0.4453,  ...,  3.7500,  0.1875, -0.2344],
        [ 0.0137, -0.0625,  0.1484,  ...,  0.0938,  0.0742,  0.1660]],
       dtype=torch.bfloat16)

: 

In [32]:
hs_hf_clean = torch.load("hf_hidden_states/0_clean.pt")
hs_hf = torch.load("hf_hidden_states/0_new.pt")

In [73]:
with open("vllm_log.log", 'a') as f:
    f.write("SECOND PASS\n")
    f.flush()

In [74]:
def clear_hooks(module):
    module._forward_hooks = OrderedDict()
    module._meta = {}
    
    
def add_hook(module, hook):
    module.register_forward_hook(hook)

def f(x):
    clear_hooks(x.worker.model_runner.model.model.layers[48])
    clear_hooks(x.worker.model_runner.model.model.layers[47])
    add_hook(
        x.worker.model_runner.model.model.layers[48],
        logging_hook
    )
    add_hook(
        x.worker.model_runner.model.model.layers[47],
        hook
    )

res = llm.llm_engine.collective_rpc(
    f,
)

In [75]:
res2 = llm.generate(
    prompts, sampling_params=sampling_params
)

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 4/4 [00:10<00:00,  2.74s/it, est. speed input: 804.59 toks/s, output: 182.61 toks/s]


In [76]:
print(res1[1].outputs[0].text)

 Let's see.

Illuminate requires Essence, Aura, and Nexus for the object. Let's see which objects can be illuminated.

For Block A: Essence A is present, Aura A? The initial conditions don't mention Aura A, only Aura D. So no.

For Block D: Aura D is present, but does it have essence? No, only A has essence. So can't illuminate D.

Block B: Does it have essence? No. So can't illuminate B.

Block C: No essence or aura. So the only possible Illuminate is on Block A? Wait, Block A has essence and aura? Wait, the initial conditions say "aura Block D", so Aura D is true, but Aura A is not mentioned. So no.

Hmm, so maybe the only possible action initially is Divest A from B.

Let me try that.

First action: Divest Block A from Block B.

Preconditions: Object (A) harmonizes Block B (yes), Essence A (yes), Nexus (yes). So that's okay.

Effects:

- Pulse A becomes true.

- Essence Block B becomes true (since the other object is B, so Essence other (B) becomes true).

- The following are remove

In [ ]:
print(res2[1].outputs[0].text)

 Let's see.

Illuminate requires Essence of the object, its aura, and nexus. Let's see which objects can be illuminated.

- Block A: has essence (yes), aura? The initial conditions don't mention aura A, only aura D. So no aura for A. So can't illuminate A.

- Block B: essence? No, only A has essence. So can't illuminate B.

- Block C: essence? No. So can't illuminate C.

- Block D: has aura (yes), but does it have essence? No. So can't illuminate D.

Hmm, so the only possible action with Divest is Divest A from B. Let's try that.

First step: Divest A from B.

Effects of Divest A from B:

- The following become true: Pulse A, Essence B.

- The following become false: A harmonizes B (so that harmony is removed), Essence A, Nexus.

So after Divest A from B:

- Essence B is now present.

- Pulse A is present.

- Nexus is gone (since it was a precondition and is removed).

- The harmony A-B is removed.

- Essence A is gone.

So now, the state is:

- Essence B (from Divest's effect)

- Puls

: 

In [49]:
??llm.collective_rpc(lambda x: print(x))

Object `llm.collective_rpc(lambda x: print(x))` not found.


In [52]:

llm.llm_engine.collective_rpc(lambda x: 1/0)

Exception: Call to collective_rpc method failed: division by zero